In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
sdf = spark.table("default.car_price_train")
df = sdf.toPandas()
num_cols = [c for c in df.select_dtypes(include='number').columns if c not in ['price','car_id']]
df2 = df[num_cols + ['price']].dropna()
X = df2[num_cols]
y = df2['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train:", len(X_train), "| Test:", len(X_test))
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
lr_r2 = r2_score(y_test, y_pred_lr)
lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(lr_mse)
coef_df = pd.DataFrame({'Feature': num_cols, 'Coefficient': lr.coef_}).sort_values('Coefficient', key=abs, ascending=False)
print("=" * 60)
print("LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R-squared (R2):  {lr_r2:.4f}")
print(f"MSE:             {lr_mse:.2f}")
print(f"RMSE:            {lr_rmse:.2f}")
print(f"MAE:             {lr_mae:.2f}")
print(f"Intercept:       {lr.intercept_:.2f}")
print("Top 10 Feature Coefficients:")
print(coef_df.head(10).to_string(index=False))
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
dt_r2 = r2_score(y_test, y_pred_dt)
dt_mse = mean_squared_error(y_test, y_pred_dt)
dt_mae = mean_absolute_error(y_test, y_pred_dt)
dt_rmse = np.sqrt(dt_mse)
feat_imp = pd.DataFrame({'Feature': num_cols, 'Importance': dt.feature_importances_}).sort_values('Importance', ascending=False)
print("=" * 60)
print("DECISION TREE REGRESSOR RESULTS")
print("=" * 60)
print(f"R-squared (R2):  {dt_r2:.4f}")
print(f"MSE:             {dt_mse:.2f}")
print(f"RMSE:            {dt_rmse:.2f}")
print(f"MAE:             {dt_mae:.2f}")
print("Max Depth: 5")
print("Top 10 Feature Importances:")
print(feat_imp.head(10).to_string(index=False))
models = ['Linear Regression', 'Decision Tree (depth=5)']
comparison = pd.DataFrame({'Model': models, 'R2': [lr_r2, dt_r2], 'MSE': [lr_mse, dt_mse], 'RMSE': [lr_rmse, dt_rmse], 'MAE': [lr_mae, dt_mae]})
print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(comparison.to_string(index=False))

Train: 121 | Test: 31
LINEAR REGRESSION RESULTS
R-squared (R2):  0.9936
MSE:             226113.39
RMSE:            475.51
MAE:             348.81
Intercept:       12620.12
Top 10 Feature Coefficients:
            Feature  Coefficient
   price_per_weight  4462.771950
    power_to_weight -2128.999647
         enginesize  1885.170587
          hp_per_cc  1432.429799
compression_per_cyl  -899.221429
         horsepower   885.326853
         curbweight   735.315318
            citympg   640.963460
   compressionratio   620.458329
    brand_avg_price   566.426792
DECISION TREE REGRESSOR RESULTS
R-squared (R2):  0.8771
MSE:             4318768.05
RMSE:            2078.16
MAE:             1478.92
Max Depth: 5
Top 10 Feature Importances:
         Feature  Importance
price_per_weight    0.697459
 brand_avg_price    0.193568
      curbweight    0.095347
         avg_mpg    0.005906
       carlength    0.001824
       boreratio    0.001477
       symboling    0.001122
      enginesize    0.000886

In [0]:
import r2_score, mean_squared_error, mean_absolute_error

In [0]:
sdf = spark.table("default.car_price_train")
df = sdf.toPandas()
print("Shape:", df.shape)
print("Columns:", list(df.columns))
num_cols = df.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['price','car_id']]
df2 = df[num_cols + ['price']].dropna()
print("Features:", len(num_cols), "| Rows:", len(df2))

Shape: (152, 39)
Columns: ['car_id', 'symboling', 'carname', 'fueltype', 'aspiration', 'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase', 'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype', 'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke', 'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg', 'price', 'brand', 'power_to_weight', 'hp_per_cc', 'avg_mpg', 'hp_bin', 'engine_category', 'brand_avg_price', 'carbody_popularity', 'drivewheel_freq', 'cyl_num', 'compression_per_cyl', 'bore_stroke_ratio', 'price_per_weight']
Features: 24 | Rows: 152


In [0]:
numeric_cols = ['symboling','wheelbase','carlength','carwidth','carheight','curbweight','enginesize','boreratio','stroke','compressionratio','horsepower','peakrpe','citympg','highwaympg','power_to_weight','hp_per_cc','avg_mpg','brand_avg_price','carbody_popularity','drivewheel_freq','cyl_num','compression_per_cyl','bore_stroke_ratio','price_per_weight']
target = 'price'
df2 = df[numeric_cols + [target]].dropna()
print("Clean shape:", df2.shape)

---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File <command-4888923945615822>, line 3
      1 numeric_cols = ['symboling','wheelbase','carlength','carwidth','carheight','curbweight','enginesize','boreratio','stroke','compressionratio','horsepower','peakrpe','citympg','highwaympg','power_to_weight','hp_per_cc','avg_mpg','brand_avg_price','carbody_popularity','drivewheel_freq','cyl_num','compression_per_cyl','bore_stroke_ratio','price_per_weight']
      2 target = 'price'
----> 3 df2 = df[numeric_cols + [target]].dropna()
      4 print("Clean shape:", df2.shape)

File /databricks/python/lib/python3.12/site-packages/pandas/core/frame.py:4108, in DataFrame.__getitem__(self, key)
   4106     if is_iterator(key):
   4107         key = list(key)
-> 4108     indexer = self.columns._get_indexer_strict(key, "columns")[1]
   4110 # take() does not accept boolean indexers
   4111 if getattr(in

In [0]:
 len(X_te))

  File <command-4888923945615852>, line 1
    len(X_te))
             ^
SyntaxError: unmatched ')'


In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
X = df2[num_cols]
y = df2['price']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train size:", len(X_train), "| Test size:", len(X_test))

Train size: 121 | Test size: 31


In [0]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
y_pred_lr = lr.predict(X_test_s)
lr_r2 = r2_score(y_test, y_pred_lr)
lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(lr_mse)
coef_df = pd.DataFrame({'Feature': num_cols, 'Coefficient': lr.coef_}).sort_values('Coefficient', key=abs, ascending=False)
print("=" * 60)
print("LINEAR REGRESSION RESULTS")
print("=" * 60)
print(f"R-squared (R2):  {lr_r2:.4f}")
print(f"MSE:             {lr_mse:.2f}")
print(f"RMSE:            {lr_rmse:.2f}")
print(f"MAE:             {lr_mae:.2f}")
print(f"Intercept:       {lr.intercept_:.2f}")
print("\nTop 10 Coefficients:")
print(coef_df.head(10).to_string(index=False))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4888923945615846>, line 8
      6 lr_mae = mean_absolute_error(y_test, y_pred_lr)
      7 lr_rmse = np.sqrt(lr_mse)
----> 8 coef_df = pd.DataFrame({'Feature': num_cols, 'Coefficient': lr.coef_}).sort_values('Coefficient', key=abs, ascending=False)
      9 print("=" * 60)
     10 print("LINEAR REGRESSION RESULTS")

NameError: name 'pd' is not defined

In [0]:
print("Cols:", list(df.columns))
print(df.dtypes)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4888923945615829>, line 1
----> 1 print("Cols:", list(df.columns))
      2 print(df.dtypes)

NameError: name 'df' is not defined

# 🚗 Car Price Estimation — Regression Analysis
## Linear Regression & Decision Tree Regressor
### Technical Analysis with Coefficients, R², MSE and MAE
**Author:** Data Science Team | **Date:** 2026-04-08 | **Framework:** PySpark + Scikit-learn

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [0]:
spark.sql("SHOW TABLES IN default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|car_price_assignment|      false|
| default|     car_price_clean|      false|
| default|  car_price_featured|      false|
| default|       car_price_raw|      false|
| default|      car_price_test|      false|
| default|     car_price_train|      false|
| default|       flights_spark|      false|
| default|        hotels_spark|      false|
| default|     sessions_joined|      false|
| default|      sessions_spark|      false|
| default|         users_spark|      false|
+--------+--------------------+-----------+



In [0]:
df = spark_df.toPandas()
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print(df.dtypes)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4888923945615797>, line 1
----> 1 df = spark_df.toPandas()
      2 print("Shape:", df.shape)
      3 print("Columns:", list(df.columns))

NameError: name 'spark_df' is not defined

In [0]:
df = spark_df.toPandas()
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5545332807272489>, line 1
----> 1 df = spark_df.toPandas()
      2 print("Dataset shape:", df.shape)
      3 print("Columns:", list(df.columns))

NameError: name 'spark_df' is not defined

In [0]:
spark_df = spark.read.csv("dbfs:/user/hive/warehouse/", header=True, inferSchema=True)
display(spark_df.limit(3))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-5371689734724432>, line 1
----> 1 files = dbutils.fs.ls("dbfs:/FileStore/")
      2 for f in files[:15]:
      3     print(f.name, f.size)

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:56, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     52     pass
     54 error_exception = ExecutionError(str(e))
---> 56 raise patch_exception_with_error_details(
     57     error_exception,
     58     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     59 ) from None

ExecutionError: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /FileStore SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptio